<div style="border-left: 5px solid #4285f4; padding: 16px 20px; margin: 6px 0 20px 0; border-radius: 6px; background: rgba(66,133,244,0.06);">

<h1>🩺 Prueba previa — diagnóstico de Colab</h1>

<p><strong>Compañero de <code>Mejorar_Imagenes_con_IA.ipynb</code>.</strong> Comprueba el entorno, los recursos externos y el estado de una preparación anterior. Es de <strong>solo lectura</strong>: no instala paquetes ni modifica el proyecto.</p>

<hr>

<h3>📋 Cómo usarlo</h3>

<ol>
<li>Sube este cuaderno a Colab y elige <em>Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)</em>.</li>
<li>Ejecuta <strong>todo</strong> el cuaderno con <em>Entorno de ejecución → Ejecutar todas</em>.</li>
<li>Al final se guarda un informe <code>.txt</code>: descárgalo y adjúntalo si algo falla.</li>
<li>Después abre y ejecuta <code>Mejorar_Imagenes_con_IA.ipynb</code>.</li>
</ol>

<p><strong>💡 Nota:</strong> si lo ejecutas en local (sin Colab), las rutas <code>/content</code> no existirán y el informe se guardará en la carpeta temporal; el resto de comprobaciones sigue funcionando.</p>

<p><strong>🔁 Repetible:</strong> puedes ejecutarlo antes y después de la preparación del cuaderno principal (celda 2); cada ejecución guarda un <code>.txt</code> con su hora. No monta Drive para no pedir permisos, así que la sección 5 solo verá el estado de Drive si ya está montado.</p>

</div>


In [ ]:
# Diagnóstico de solo lectura: no instala paquetes ni modifica el proyecto.
# Ejecuta todo el cuaderno ANTES de probar Mejorar_Imagenes_con_IA.ipynb.

import hashlib
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

INICIO_UTC = datetime.now(timezone.utc)
REPORTE = []

PROYECTO = Path("/content/Mejorar_Imagenes_con_IA")
DRIVE_PROYECTO = Path("/content/drive/MyDrive/Mejorar_Imagenes_con_IA")


def log(texto=""):
    """Imprime la línea y la guarda para el informe final."""
    print(texto)
    REPORTE.append(str(texto))


def seccion(titulo):
    log()
    log("=" * 64)
    log(titulo)
    log("=" * 64)


def formato_bytes(cantidad):
    cantidad = float(cantidad)
    for unidad in ("B", "KB", "MB", "GB", "TB"):
        if cantidad < 1024 or unidad == "TB":
            return f"{cantidad:.1f} {unidad}"
        cantidad /= 1024


def sha256_archivo(ruta, bloque=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for trozo in iter(lambda: archivo.read(bloque), b""):
            digest.update(trozo)
    return digest.hexdigest()


seccion("1. Sesión y sistema")
log(f"Fecha (UTC): {INICIO_UTC:%Y-%m-%d %H:%M:%S}")
log(f"Python: {sys.version.split()[0]}")
log(f"Plataforma: {platform.platform()}")
log(f"CPUs lógicas: {os.cpu_count()}")

try:
    import psutil

    memoria = psutil.virtual_memory()
    log(f"RAM: {formato_bytes(memoria.total)} (disponible {formato_bytes(memoria.available)})")
except Exception as error:
    log(f"RAM: no se pudo consultar ({error!r})")

for ruta in ("/content", "/content/drive"):
    try:
        uso = shutil.disk_usage(ruta)
        log(f"Disco {ruta}: {formato_bytes(uso.free)} libres de {formato_bytes(uso.total)}")
    except Exception:
        if ruta == "/content/drive":
            log(f"Disco {ruta}: no montado (este cuaderno no monta Drive)")
        else:
            log(f"Disco {ruta}: no disponible")

seccion("2. GPU y PyTorch")
try:
    import torch

    log(f"PyTorch: {torch.__version__}")
    log(f"CUDA compilada en PyTorch: {torch.version.cuda}")
    log(f"CUDA disponible: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        propiedades = torch.cuda.get_device_properties(0)
        log(f"GPU: {propiedades.name}")
        log(f"Memoria de la GPU: {formato_bytes(propiedades.total_memory)}")
        log(f"Capacidad de cómputo: {propiedades.major}.{propiedades.minor}")
        try:
            bf16 = torch.cuda.is_bf16_supported()
            log(f"bfloat16 soportado: {bf16}")
            if not bf16:
                log("AVISO: HYPIR usa bfloat16; en esta GPU podría fallar. Prueba un runtime L4/A100.")
        except Exception as error:
            log(f"bfloat16: no se pudo consultar ({error!r})")
    else:
        log("AVISO: no hay GPU. Usa Entorno de ejecución → Cambiar tipo de entorno → GPU.")
except Exception as error:
    log(f"PyTorch: ERROR {error!r}")

try:
    salida_nvidia = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        timeout=20,
    )
    if salida_nvidia.returncode == 0 and salida_nvidia.stdout.strip():
        for linea in salida_nvidia.stdout.strip().splitlines():
            log(f"nvidia-smi: {linea.strip()}")
    else:
        log("nvidia-smi: no disponible en este entorno")
except Exception as error:
    log(f"nvidia-smi: no disponible ({error!r})")


In [ ]:
seccion("3. Diagnóstico del proyecto (diagnostico_entorno.py)")

if Path("/content").is_dir():
    CARPETA_SCRIPT = Path("/content")
else:
    CARPETA_SCRIPT = Path(tempfile.gettempdir()) / "diagnostico_colab"
CARPETA_SCRIPT.mkdir(parents=True, exist_ok=True)
RUTA_SCRIPT = CARPETA_SCRIPT / "diagnostico_entorno.py"

# Copia embebida del script del repositorio: así el informe es reproducible
# aunque el archivo no se haya subido a la sesión.

SCRIPT_DIAGNOSTICO = r'''"""
DIAGNÓSTICO DE ENTORNO — Mejorar_Imagenes_con_IA
Pegar el bloque marcado en una celda nueva de Google Colab, en la sesión
conocida como funcional. Este script solo informa; no instala ni modifica nada.
"""
# --- DIAGNÓSTICO DE ENTORNO: pegar en una celda de Colab ---
import sys
import platform
import importlib.metadata as metadata

print("Python:", sys.version.replace("\n", " "))
print("Plataforma:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    print("CUDA compilada en PyTorch:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch: ERROR", repr(e))

packages = [
    # Stack que ya aporta Google Colab; el cuaderno no lo reinstala.
    "torch", "torchvision", "torchaudio",
    # Requisitos que el cuaderno instala para el flujo CodeFormer.
    "numpy", "Pillow", "opencv-python", "opencv-python-headless",
    "scipy", "scikit-image", "tqdm", "yapf", "lmdb", "pyyaml",
    "addict", "future", "requests", "tb-nightly", "lpips", "gdown",
    # Requisitos que el cuaderno instala para el flujo HYPIR.
    "accelerate", "diffusers", "transformers", "peft", "einops",
    "omegaconf", "timm", "huggingface_hub",
    # Interfaz del cuaderno y utilidades de Colab.
    "matplotlib", "ipywidgets",
]
# Nota: "basicsr", "facexlib", "gfpgan" y "realesrgan" NO se instalan como
# paquetes en este proyecto. "basicsr" y "facelib" forman parte del código
# fuente de CodeFormer y el cuaderno los importa desde esa carpeta.
print("\nVersiones de paquetes:")
for package in packages:
    try:
        print(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: no instalado (o nombre de distribución distinto)")
    except Exception as e:
        print(f"{package}: error al consultar ({e!r})")

print("\nRutas relevantes:")
from pathlib import Path
for p in [
    Path("/content/Mejorar_Imagenes_con_IA/CodeFormer"),
    Path("/content/CodeFormer"),
    Path("/content/drive/MyDrive/Mejorar_Imagenes_con_IA/CodeFormer"),
]:
    print(f"{p}: existe={p.exists()} ; inference_codeformer.py={(p / 'inference_codeformer.py').exists()}")
    print(
        "    módulos locales: "
        f"basicsr={(p / 'basicsr' / '__init__.py').exists()} ; "
        f"facelib={(p / 'facelib' / '__init__.py').exists()} ; "
        f"pesos/facelib={(p / 'weights' / 'facelib').is_dir()}"
    )

# Opcional: guardar el diagnóstico en un archivo descargable desde Colab.
salida = Path("/content/diagnostico_entorno_mejorar_imagenes.txt")
try:
    salida.write_text(
        "Python: " + sys.version.replace("\n", " ") + "\n"
        + "Plataforma: " + platform.platform() + "\n",
        encoding="utf-8"
    )
    print("\nResumen básico guardado en:", salida)
except Exception as e:
    print("No se pudo guardar el resumen:", repr(e))
# --- FIN DEL DIAGNÓSTICO ---
'''


RUTA_SCRIPT.write_text(SCRIPT_DIAGNOSTICO, encoding="utf-8", newline="\n")
log(f"Script escrito en: {RUTA_SCRIPT}")
log(f"SHA-256 del script: {sha256_archivo(RUTA_SCRIPT)}")
log("")

try:
    proceso = subprocess.run(
        [sys.executable, str(RUTA_SCRIPT)],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=180,
    )
    salida = proceso.stdout or ""
    print(salida, end="" if salida.endswith("\n") else "\n")
    REPORTE.extend(salida.rstrip("\n").splitlines())
    if proceso.stderr.strip():
        log("")
        log("--- stderr del diagnóstico ---")
        for linea in proceso.stderr.rstrip("\n").splitlines():
            log(linea)
    log(f"Código de salida del script: {proceso.returncode}")
except Exception as error:
    log(f"No se pudo ejecutar el diagnóstico: {error!r}")


In [ ]:
seccion("4. Conectividad con los recursos externos")

BASE_RELEASE = (
    "https://github.com/Pablitus666/Mejorar_Imagenes_con_IA/"
    "releases/download/modelos-v1/"
)
RECURSOS = [
    ("codeformer.pth", BASE_RELEASE + "codeformer.pth", 100_000_000, False),
    ("detection_Resnet50_Final.pth", BASE_RELEASE + "detection_Resnet50_Final.pth", 1_000_000, False),
    ("parsing_parsenet.pth", BASE_RELEASE + "parsing_parsenet.pth", 1_000_000, False),
    ("RealESRGAN_x2plus.pth", BASE_RELEASE + "RealESRGAN_x2plus.pth", 60_000_000, False),
    ("yolov5l-face.pth", BASE_RELEASE + "yolov5l-face.pth", 100_000_000, False),
    ("HYPIR código (b61d107)", "https://github.com/XPixelGroup/HYPIR/archive/b61d107.zip", 1_000_000, False),
    (
        "HYPIR_sd2.pth",
        "https://huggingface.co/lxq007/HYPIR/resolve/"
        "93e4cceb10df58bb5d2441159f6abd4cf56430ba/HYPIR_sd2.pth",
        500_000_000,
        False,
    ),
    (
        "SD 2.1 (mirror STHCD)",
        "https://huggingface.co/STHCD/stable-diffusion-2-1-base/resolve/main/model_index.json",
        None,
        False,
    ),
    (
        "SD 2.1 oficial (401 esperado)",
        "https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/model_index.json",
        None,
        True,
    ),
]


def comprobar_recurso(etiqueta, url, minimo, espera_401):
    inicio = time.perf_counter()
    peticion = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "diagnostico-colab"})
    try:
        with urllib.request.urlopen(peticion, timeout=30) as respuesta:
            estado = respuesta.status
            largo = respuesta.headers.get("Content-Length")
    except urllib.error.HTTPError as error:
        estado = error.code
        largo = None
    except Exception as error:
        log(f"  FALLO  {etiqueta}: {type(error).__name__}: {error}")
        return False

    segundos = time.perf_counter() - inicio

    if espera_401 and estado == 401:
        log(f"  OK     {etiqueta}: HTTP 401 (protegido, se usa el mirror) — {segundos:.1f} s")
        return True

    detalle = f"HTTP {estado}"
    if largo is not None:
        detalle += f", {formato_bytes(int(largo))}"
        if minimo and int(largo) < minimo:
            detalle += f" — ¡MENOR que el mínimo esperado ({formato_bytes(minimo)})!"
    if estado == 200:
        log(f"  OK     {etiqueta}: {detalle} — {segundos:.1f} s")
        return True
    log(f"  FALLO  {etiqueta}: {detalle} — {segundos:.1f} s")
    return False


fallos = 0
try:
    import urllib.error
    import urllib.request

    for etiqueta, url, minimo, espera_401 in RECURSOS:
        if not comprobar_recurso(etiqueta, url, minimo, espera_401):
            fallos += 1
    log("")
    log(f"Recursos con problemas: {fallos} de {len(RECURSOS)}")
except Exception as error:
    log(f"No se pudo comprobar la conectividad: {error!r}")


In [ ]:
seccion("5. Estado de una preparación anterior")

VERIFICAR_HASHES = False  # Ponlo en True para releer los pesos y comparar SHA-256 (lento).

CANDIDATOS_CODEFORMER = [
    PROYECTO / "CodeFormer",
    Path("/content/CodeFormer"),
    DRIVE_PROYECTO / "CodeFormer",
]
REPO_CODEFORMER = next(
    (ruta for ruta in CANDIDATOS_CODEFORMER if (ruta / "inference_codeformer.py").is_file()),
    None,
)

if REPO_CODEFORMER is None:
    log("CodeFormer: no se encontró un repositorio preparado (normal antes de ejecutar la celda 2).")
else:
    log(f"CodeFormer: {REPO_CODEFORMER}")
    log("  inference_codeformer.py: presente")
    for modulo in ("basicsr", "facelib"):
        presente = (REPO_CODEFORMER / modulo / "__init__.py").is_file()
        log(f"  módulo {modulo}: {'presente' if presente else 'FALTA'}")
    log(f"  pesos/facelib: {'presente' if (REPO_CODEFORMER / 'weights' / 'facelib').is_dir() else 'FALTA'}")

    PESOS_CODEFORMER = [
        ("codeformer.pth", Path("weights") / "CodeFormer" / "codeformer.pth",
         100_000_000, "1009e537e0c2a07d4cabce6355f53cb66767cd4b4297ec7a4a64ca4b8a5684b7"),
        ("detection_Resnet50_Final.pth", Path("weights") / "facelib" / "detection_Resnet50_Final.pth",
         1_000_000, "6d1de9c2944f2ccddca5f5e010ea5ae64a39845a86311af6fdf30841b0a5a16d"),
        ("parsing_parsenet.pth", Path("weights") / "facelib" / "parsing_parsenet.pth",
         1_000_000, "3d558d8d0e42c20224f13cf5a29c79eba2d59913419f945545d8cf7b72920de2"),
        ("RealESRGAN_x2plus.pth", Path("weights") / "realesrgan" / "RealESRGAN_x2plus.pth",
         60_000_000, "49fafd45f8fd7aa8d31ab2a22d14d91b536c34494a5cfe31eb5d89c2fa266abb"),
        ("yolov5l-face.pth", Path("weights") / "facelib" / "yolov5l-face.pth",
         100_000_000, "1ba9d2125fda4d823df5152b9fc2903c59aa76c0d3771e02bcf13a56a282cf96"),
    ]
    for nombre, relativa, minimo, esperado in PESOS_CODEFORMER:
        ruta = REPO_CODEFORMER / relativa
        if not ruta.is_file():
            log(f"  {nombre}: FALTA")
            continue
        bytes_peso = ruta.stat().st_size
        estado = "OK" if bytes_peso >= minimo else "TRUNCADO"
        log(f"  {nombre}: {formato_bytes(bytes_peso)} — {estado}")
        if VERIFICAR_HASHES:
            obtenido = sha256_archivo(ruta)
            log(f"      sha256 {'coincide' if obtenido == esperado else 'NO coincide'}: {obtenido}")

for marcador in (PROYECTO / "cache" / "codeformer_commit.txt",
                 DRIVE_PROYECTO / "cache" / "codeformer_commit.txt"):
    if marcador.is_file():
        log(f"Commit de CodeFormer fijado: {marcador.read_text(encoding='utf-8').strip()}")
        break
else:
    log("Commit de CodeFormer: aún no registrado")

repo_hypir = PROYECTO / "HYPIR"
log(f"HYPIR: {'preparado' if (repo_hypir / 'test.py').is_file() else 'no preparado'}")
marcador_hypir = repo_hypir / ".mejorar_hypir_revision"
if marcador_hypir.is_file():
    log(f"  revisión fijada: {marcador_hypir.read_text(encoding='utf-8').strip()}")

for candidato in (PROYECTO / "modelos" / "HYPIR_sd2.pth",
                  DRIVE_PROYECTO / "modelos" / "HYPIR_sd2.pth"):
    if candidato.is_file():
        bytes_peso = candidato.stat().st_size
        log(f"Peso HYPIR_sd2.pth: {formato_bytes(bytes_peso)} en {candidato}")
        if VERIFICAR_HASHES:
            obtenido = sha256_archivo(candidato)
            esperado = "d538a2cb925451fab1f75adfe715ac2b1c8bb12fa32a0851ba01be2347b354d6"
            log(f"  sha256 {'coincide' if obtenido == esperado else 'NO coincide'}: {obtenido}")
        break
else:
    log("Peso HYPIR_sd2.pth: aún no descargado")

cache_sd = PROYECTO / "_cache_local" / "stable-diffusion-2-1-base"
if (cache_sd / ".mejorar_cache_completa").is_file():
    archivos = [ruta for ruta in cache_sd.rglob("*") if ruta.is_file()]
    total = sum(ruta.stat().st_size for ruta in archivos)
    log(f"Caché local SD 2.1: completa ({len(archivos)} archivos, {formato_bytes(total)})")
else:
    log("Caché local SD 2.1: no preparada")

for revision in (PROYECTO / "cache" / "stable-diffusion-2-1-base.revision",
                 DRIVE_PROYECTO / "cache" / "stable-diffusion-2-1-base.revision"):
    if revision.is_file():
        log(f"Revisión SD 2.1 fijada: {revision.read_text(encoding='utf-8').strip()}")
        break

marcas_deps = sorted(Path("/content").glob(".mejorar_*deps_py*"))
if marcas_deps:
    log("Marcadores de dependencias: " + ", ".join(marca.name for marca in marcas_deps))
else:
    log("Marcadores de dependencias: ninguno (aún no se ejecutó la celda 2)")

if not DRIVE_PROYECTO.parent.is_dir():
    log("Drive no montado: el estado anterior solo se puede ver en /content.")


In [ ]:
# Esta celda se puede repetir sin duplicar el informe: imprime el guardado
# pero no añade líneas nuevas a REPORTE.
texto = "\n".join(REPORTE).lstrip("\n") + "\n"
marca = INICIO_UTC.strftime("%Y%m%d_%H%M%S")
nombre = f"diagnostico_colab_{marca}.txt"

print()
print("=" * 64)
print("6. Guardar el informe")
print("=" * 64)
print(f"Líneas recopiladas: {len(REPORTE)}")

destinos = []
if Path("/content").is_dir():
    destinos.append(Path("/content") / nombre)
if DRIVE_PROYECTO.is_dir():
    destinos.append(DRIVE_PROYECTO / "diagnosticos" / nombre)
if not destinos:
    destinos.append(Path(tempfile.gettempdir()) / nombre)

for destino in destinos:
    try:
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_text(texto, encoding="utf-8")
        print(f"Informe guardado en: {destino} ({formato_bytes(destino.stat().st_size)})")
    except Exception as error:
        print(f"No se pudo guardar en {destino}: {error!r}")

DESCARGAR = False  # Cámbialo a True para descargar el informe automáticamente en Colab.
if DESCARGAR:
    try:
        from google.colab import files

        files.download(str(destinos[0]))
    except Exception as error:
        print(f"Descarga no disponible: {error!r}")


---

<h2>📄 Qué hacer con el informe</h2>

Adjunta el `.txt` (o copia su contenido) junto con:

1. El **traceback completo** del error, si lo hubo.
2. La **celda** del cuaderno principal donde ocurrió (0–12).
3. El **nombre de la GPU** que muestra este informe.

<h2>🔎 Cómo leerlo</h2>

<ul>
<li><strong>Sección 2:</strong> confirma que hay GPU y si admite <code>bfloat16</code> (lo usa HYPIR).</li>
<li><strong>Sección 3:</strong> versiones instaladas; compara con las fijadas por el cuaderno.</li>
<li><strong>Sección 4:</strong> los 9 recursos deben salir <code>OK</code>; el 401 de Stability es esperado.</li>
<li><strong>Sección 5:</strong> distingue «aún no preparado» (normal antes de la celda 2) de «FALTA» o «TRUNCADO».</li>
</ul>

<p><strong>💡 Después de la preparación:</strong> vuelve a ejecutar el cuaderno (o al menos las secciones 3 y 5) para
dejar constancia de las versiones finales y del estado real de CodeFormer/HYPIR/SD 2.1. El informe anterior
será el «antes» y el nuevo el «después».</p>
